# Phase 4 — Dimensional Modeling

The cleaned transaction data is transformed into a star-schema model for downstream SQL analysis and reporting.

The model preserves the transaction-line-item grain identified during data-quality assessment.

## 4.1 Objective

Build a reusable analytical model that separates transaction facts from descriptive dimensions while preserving the original transaction-line-item grain.

### Load Cleaned Dataset and Restore the Data Types

The cleaned dataset is stored as a CSV in the previous ETL phase. While CSV preserves the cleaned values and structure of the dataset, it does not retain pandas-specific data types. As a result, the `date` column is read back as text (`object`) by default.

To restore the original schema, the `date` column is parsed as a datetime during import. This ensures that datetime operations can be performed without requiring additional transformations later in the project.

In [2]:
import pandas as pd
from pathlib import Path

In [3]:
# Path to the cleaned dataset generated during the ETL phase
input_path = Path("../data/processed/scanner_data_clean.csv")

In [4]:
df = pd.read_csv(
    input_path,
    parse_dates=['date']
)

## 4.2 Dimensional Modeling

Dimensional modeling separates measurable business events from the descriptive information used to analyze them.

For this project:

- `fact_sales` stores transaction-line-item measures.
- `dim_date` stores calendar attributes.
- `dim_customer` stores customer attributes.
- `dim_product` stores product attributes.

A star schema is used so the fact table can be analyzed efficiently across these dimensions.

## 4.3 Why a Star Schema?

A star schema is the most widely used dimensional model in analytical databases and data warehouses.

It consists of a central **fact table** surrounded by multiple **dimension tables**, creating a structure that resembles a star.

For this retail sales dataset, the star schema separates transactional measures such as quantity and sales amount from descriptive information such as customers, products, and dates.


### Star Schema Overview

```text
                 dim_date
                     │
                     │
dim_customer ─── fact_sales ─── dim_product
```

- **Fact Table:** Stores measurable business events.
- **Dimension Tables:** Store descriptive attributes that provide context for those events.
```

## 4.4 Identify the Business Process

Inspection of the cleaned dataset shows that each record captures the sale of a product to a customer during a retail transaction.

**Business Process:** Retail Sales Transactions


In [5]:
df.head()

,date,customer_id,transaction_id,sku_category,sku,quantity,sales_amount
0,2016-01-02,2547,1,X52,0EM7L,1.0,3.13
1,2016-01-02,822,2,2ML,68BRQ,1.0,5.46
2,2016-01-02,3686,3,0H2,CZUZX,1.0,6.35
3,2016-01-02,3719,4,0H2,549KK,1.0,5.59
4,2016-01-02,9200,5,0H2,K8EHH,1.0,6.88


In [6]:
df.columns

Index(['date', 'customer_id', 'transaction_id', 'sku_category', 'sku',
       'quantity', 'sales_amount'],
      dtype='object')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131706 entries, 0 to 131705
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   date            131706 non-null  datetime64[ns]
 1   customer_id     131706 non-null  int64         
 2   transaction_id  131706 non-null  int64         
 3   sku_category    131706 non-null  object        
 4   sku             131706 non-null  object        
 5   quantity        131706 non-null  float64       
 6   sales_amount    131706 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(2), object(2)
memory usage: 7.0+ MB


## 4.5 Define the Grain

The grain defines the level of detail represented by each record in the fact table and serves as the foundation for the dimensional model.

Analysis of the cleaned dataset showed that a single `transaction_id` may contain multiple products, with each row representing a unique product purchased within the transaction. Therefore, the dataset is recorded at the **transaction-line-item level** rather than the transaction level.

> **One row represents one SKU purchased within one customer transaction on one date.**


In [8]:
# Number of line items in each transaction
df.groupby('transaction_id').size().value_counts().sort_index()

1     32906
2     16845
3      6802
4      3478
5      1851
6      1036
7       646
8       426
9       254
10      167
11       94
12       64
13       52
14       23
15       16
16        4
17        6
18        4
19        5
20        1
21        2
Name: count, dtype: int64

In [9]:
# Number of line items per transaction
df.groupby('transaction_id').size().sort_values(ascending=False).head()

transaction_id
1348     21
19098    21
42295    20
34204    19
16271    19
dtype: int64

In [10]:
df[df['transaction_id'] == 48]

,date,customer_id,transaction_id,sku_category,sku,quantity,sales_amount
76,2016-01-02,6112,48,0WT,AK33R,1.0,12.40
77,2016-01-02,6112,48,R6E,F64H7,1.0,2.86
78,2016-01-02,6112,48,29A,RH5J5,1.0,4.23



The results confirm that a single transaction can contain multiple purchased products. Each row therefore represents an individual transaction line item rather than an entire transaction.


## 4.6 Design the Analytical Data Model

This involves identifying the measurable business facts, the descriptive business dimensions, and the analytical tables required to support reporting and business intelligence. The resulting design will serve as the blueprint for constructing the star schema in the following sections.

###  Identify Facts and Dimensions

Each attribute in the cleaned dataset was classified according to its role in the analytical model.

- **Facts** represent measurable business events that can be aggregated for analysis.
- **Dimensions** provide descriptive information that gives context to those business events.

This classification forms the foundation of the dimensional model and determines how the dataset will be organized into fact and dimension tables.

In [11]:
df.info()
df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131706 entries, 0 to 131705
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   date            131706 non-null  datetime64[ns]
 1   customer_id     131706 non-null  int64         
 2   transaction_id  131706 non-null  int64         
 3   sku_category    131706 non-null  object        
 4   sku             131706 non-null  object        
 5   quantity        131706 non-null  float64       
 6   sales_amount    131706 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(2), object(2)
memory usage: 7.0+ MB


Index(['date', 'customer_id', 'transaction_id', 'sku_category', 'sku',
       'quantity', 'sales_amount'],
      dtype='object')

###  Column Classification

| Column           | Classification  | Reason                                                                                  |
| ---------------- | --------------- | --------------------------------------------------------------------------------------- |
| `date`           | Dimension       | Describes when the transaction occurred                                                 |
| `customer_id`    | Dimension       | Identifies the customer                                                                 |
| `sku`            | Dimension       | Identifies the purchased product                                                        |
| `sku_category`   | Dimension       | Describes the product                                                                   |
| `quantity`       | Fact            | Quantity sold                                                                           |
| `sales_amount`   | Fact            | Revenue generated                                                                       |
| `transaction_id` | Fact Identifier | Identifies the sales transaction and links multiple line items within the same purchase |


###  Determine the Required Analytical Tables

Based on the identified facts, dimensions, and the defined grain, the analytical tables required for the dimensional model were determined.

The analytical model will consist of three dimension tables (`dim_date`, `dim_customer`, and `dim_product`) and one central fact table (`fact_sales`).



| Business Entity          | Dataset Columns                              | Analytical Table |
| ------------------------ | -------------------------------------------- | ---------------- |
| Date                     | `date`                                       | `dim_date`       |
| Customer                 | `customer_id`                                | `dim_customer`   |
| Product                  | `sku`, `sku_category`                        | `dim_product`    |
| Retail Sales Transaction | `transaction_id`, `quantity`, `sales_amount` | `fact_sales`     |


### Design the Star Schema

The identified analytical tables are organized into a star schema consisting of one central fact table surrounded by supporting dimension tables.

The fact table stores measurable retail sales events, while the dimension tables provide descriptive context that enables analysis across customers, products, and time.

```text
                 dim_date
                     │
                     │
dim_customer ─── fact_sales ─── dim_product
```

###  Define the Table Structures

Before implementing the dimensional model, the structure of each analytical table is defined. Establishing the table structures in advance ensures that every attribute is assigned to the appropriate table and that relationships remain consistent throughout the implementation.



| Table            | Planned Columns                                                                           |
| ---------------- | ----------------------------------------------------------------------------------------- |
| **dim_date**     | `date_key`, `date`, `year`, `quarter`, `month`, `month_name`, `day`, `day_name`, `week`, `is_weekend` |
| **dim_customer** | `customer_key` , `customer_id`                                                                             |
| **dim_product**  | `product_key` ,`sku`, `sku_category`                                                                     |
| **fact_sales**   | `transaction_id`, `date_key`, `customer_key`, `product_key`, `quantity`, `sales_amount`                |


## 4.7 Build the Dimension Tables

With the analytical model in place, the next step is to build the dimension tables that will provide descriptive context for retail sales transactions.

The dimension tables are built first because the fact table will use their surrogate keys.

Each dimension is created from the relevant business key and descriptive attributes in the cleaned dataset.

In this section, we'll build dimensions for **time**, **customers**, and **products**, and then validate that each one correctly represents its corresponding business entity.

###  Build `dim_date`

The cleaned dataset contains only a single `date` column. Instead of repeatedly extracting calendar information during analysis, we'll create a dedicated date dimension containing commonly used calendar attributes.

In [12]:
dim_date = df[['date']].drop_duplicates().sort_values('date').reset_index(drop=True)

In [13]:
dim_date.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 363 entries, 0 to 362
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    363 non-null    datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 3.0 KB


In [14]:
dim_date['year'] = dim_date['date'].dt.year
dim_date['quarter'] = dim_date['date'].dt.quarter
dim_date['month'] = dim_date['date'].dt.month
dim_date['month_name'] = dim_date['date'].dt.month_name()
dim_date['day'] = dim_date['date'].dt.day
dim_date['day_name'] = dim_date['date'].dt.day_name()
dim_date['week'] = dim_date['date'].dt.isocalendar().week.astype(int)
dim_date['is_weekend'] = dim_date['day_name'].isin(
    ['Saturday', 'Sunday']
)

###  Build `dim_customer`

In this dataset, the only customer-related attribute available is the customer identifier. Although simple, creating a dedicated customer dimension establishes a scalable structure that can easily accommodate additional customer attributes in the future if they become available.

In [15]:
dim_customer = df[['customer_id']].drop_duplicates().sort_values('customer_id',ascending=True).reset_index(drop=True)

###  Build `dim_product`

Products are identified by their SKU and associated category. Since the same product appears across many transactions, storing this information in a separate dimension eliminates unnecessary repetition while making product-level analysis much easier.

Each product will appear only once together with its corresponding category.

In [16]:
dim_product = df[['sku', 'sku_category']].drop_duplicates().sort_values('sku').reset_index(drop = True)

### Validate the Dimension Tables

Before building the fact table, it's important to verify that each dimension has been created correctly.


In [17]:
dim_date.head()

,date,year,quarter,month,month_name,day,day_name,week,is_weekend
0,2016-01-02,2016,1,1,January,2,Saturday,53,True
1,2016-01-03,2016,1,1,January,3,Sunday,53,True
2,2016-01-04,2016,1,1,January,4,Monday,1,False
3,2016-01-05,2016,1,1,January,5,Tuesday,1,False
4,2016-01-06,2016,1,1,January,6,Wednesday,1,False


In [18]:
dim_date.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 363 entries, 0 to 362
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        363 non-null    datetime64[ns]
 1   year        363 non-null    int32         
 2   quarter     363 non-null    int32         
 3   month       363 non-null    int32         
 4   month_name  363 non-null    object        
 5   day         363 non-null    int32         
 6   day_name    363 non-null    object        
 7   week        363 non-null    int32         
 8   is_weekend  363 non-null    bool          
dtypes: bool(1), datetime64[ns](1), int32(5), object(2)
memory usage: 16.1+ KB


In [19]:
dim_customer.head()

,customer_id
0,1
1,2
2,3
3,4
4,5


In [20]:
dim_customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22625 entries, 0 to 22624
Data columns (total 1 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  22625 non-null  int64
dtypes: int64(1)
memory usage: 176.9 KB


In [21]:
dim_product.head()

,sku,sku_category
0,00GVC,1TS
1,00OK1,J4R
2,0121I,XG4
3,01IEO,XVK
4,01IQT,Z4O


In [22]:
dim_product.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5242 entries, 0 to 5241
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   sku           5242 non-null   object
 1   sku_category  5242 non-null   object
dtypes: object(2)
memory usage: 82.0+ KB


In [23]:
print("Unique dates:", df['date'].nunique())
print("Rows in dim_date:", len(dim_date))

print("Unique customers:", df['customer_id'].nunique())
print("Rows in dim_customer:", len(dim_customer))

print("Unique products:", df['sku'].nunique())
print("Rows in dim_product:", len(dim_product))

Unique dates: 363
Rows in dim_date: 363
Unique customers: 22625
Rows in dim_customer: 22625
Unique products: 5242
Rows in dim_product: 5242


In [24]:
print("Duplicate dates:", dim_date.duplicated().sum())
print("Duplicate customers:", dim_customer.duplicated().sum())
print("Duplicate products:", dim_product.duplicated().sum())

Duplicate dates: 0
Duplicate customers: 0
Duplicate products: 0




The validation confirms that each dimension accurately represents its corresponding business entity.

| Dimension | Validation |
|-----------|------------|
| **dim_date** | One record per unique transaction date |
| **dim_customer** | One record per unique customer |
| **dim_product** | One record per unique SKU |
| **Duplicates** | No duplicate records identified in any dimension table |

All validation checks passed successfully, indicating that the dimension tables were created as intended.


### Add Surrogate Keys

Each dimension receives an integer surrogate key. These keys provide a consistent identifier within the analytical model and will be used to connect the dimensions to `fact_sales`.

The original business keys are retained alongside them for identification and traceability.


In [25]:
#date-dimention

dim_date = dim_date.sort_values("date").reset_index(drop=True)

dim_date.insert(
    0,
    "date_key",
    range(1, len(dim_date) + 1)
)

In [26]:
dim_date.head()

,date_key,date,year,quarter,month,month_name,day,day_name,week,is_weekend
0,1,2016-01-02,2016,1,1,January,2,Saturday,53,True
1,2,2016-01-03,2016,1,1,January,3,Sunday,53,True
2,3,2016-01-04,2016,1,1,January,4,Monday,1,False
3,4,2016-01-05,2016,1,1,January,5,Tuesday,1,False
4,5,2016-01-06,2016,1,1,January,6,Wednesday,1,False


In [27]:
print("Rows:", len(dim_date))
print("Unique date_keys:", dim_date["date_key"].nunique())
print("Missing date_keys:", dim_date["date_key"].isna().sum())

Rows: 363
Unique date_keys: 363
Missing date_keys: 0


In [28]:
##Customer Dimension
dim_customer = dim_customer.reset_index(drop=True)

dim_customer.insert(
    0,
    "customer_key",
    range(1, len(dim_customer) + 1)
)

In [29]:
dim_customer.head()

,customer_key,customer_id
0,1,1
1,2,2
2,3,3
3,4,4
4,5,5


In [30]:
print("Rows:", len(dim_customer))
print("Unique customer_keys:", dim_customer["customer_key"].nunique())
print("Missing customer_keys:", dim_customer["customer_key"].isna().sum())

Rows: 22625
Unique customer_keys: 22625
Missing customer_keys: 0


In [31]:
##Product Dimension
dim_product = dim_product.reset_index(drop=True)

dim_product.insert(
    0,  
    "product_key",
    range(1, len(dim_product) + 1)
)

In [32]:
dim_product.head()

,product_key,sku,sku_category
0,1,00GVC,1TS
1,2,00OK1,J4R
2,3,0121I,XG4
3,4,01IEO,XVK
4,5,01IQT,Z4O


In [33]:
print("Rows:", len(dim_product))
print("Unique product_keys:", dim_product["product_key"].nunique())
print("Missing product_keys:", dim_product["product_key"].isna().sum())

Rows: 5242
Unique product_keys: 5242
Missing product_keys: 0


### Result

The dimension layer of the analytical model has now been successfully implemented.

The three dimension tables provide descriptive context for retail sales transactions while eliminating unnecessary redundancy from the analytical model. These tables establish a reusable foundation for SQL analysis, KPI development, and dashboard reporting.


## 4.8 Build the Fact Table

The fact table is created from the cleaned transaction data. Surrogate keys from the dimension tables are mapped onto each transaction using the corresponding business keys.

The original business keys are used only for the lookup. The final fact table retains the surrogate keys as foreign keys to the dimension tables.

In [34]:
fact_sales = df[
    [
        'transaction_id',
        'date',
        'customer_id',
        'sku',
        'quantity',
        'sales_amount'
    ]
].copy()

In [35]:
fact_sales['date_key'] = (
    fact_sales['date']
    .map(dim_date.set_index('date')['date_key'])
)

fact_sales['customer_key'] = (
    fact_sales['customer_id']
    .map(dim_customer.set_index('customer_id')['customer_key'])
)

fact_sales['product_key'] = (
    fact_sales['sku']
    .map(dim_product.set_index('sku')['product_key'])
)

In [36]:
fact_sales = fact_sales[
    [
        'transaction_id',
        'date_key',
        'customer_key',
        'product_key',
        'quantity',
        'sales_amount'
    ]
].copy()

In [37]:
fact_sales.head()

,transaction_id,date_key,customer_key,product_key,quantity,sales_amount
0,1,1,2547,64,1.0,3.13
1,2,1,822,923,1.0,5.46
2,3,1,3686,1933,1.0,6.35
3,4,1,3719,777,1.0,5.59
4,5,1,9200,2972,1.0,6.88


In [38]:
fact_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131706 entries, 0 to 131705
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   transaction_id  131706 non-null  int64  
 1   date_key        131706 non-null  int64  
 2   customer_key    131706 non-null  int64  
 3   product_key     131706 non-null  int64  
 4   quantity        131706 non-null  float64
 5   sales_amount    131706 non-null  float64
dtypes: float64(2), int64(4)
memory usage: 6.0 MB


### Validate the Fact Table

The fact table should preserve the transaction-line-item grain established during the earlier modeling stage. Validation therefore focuses on row count, uniqueness of the transaction-line-item grain, and completeness of the foreign keys.

In [39]:
print("Original rows :", len(df))
print("Fact rows     :", len(fact_sales))

Original rows : 131706
Fact rows     : 131706


In [40]:
fact_sales.duplicated(
    subset=['transaction_id', 'product_key']
).sum()

0

In [41]:
fact_sales[
    ['date_key', 'customer_key', 'product_key']
].isna().sum()

date_key        0
customer_key    0
product_key     0
dtype: int64

### Validation

The fact table retains all 131,706 transaction-line-item records from the cleaned dataset, confirming that the intended grain was preserved.

The transaction-line-item uniqueness check returned no duplicates, and no missing surrogate keys were introduced during the dimension lookups.

### Verify Referential Integrity

A valid dimensional model requires every foreign key in the fact table to have a corresponding primary key in its respective dimension table.

The following checks verify that all date, customer, and product references in `fact_sales` exist in their corresponding dimensions.

In [42]:
referential_integrity = pd.DataFrame({
    'Relationship': [
        'fact_sales → dim_date',
        'fact_sales → dim_customer',
        'fact_sales → dim_product'
    ],
    
    'Missing References': [
        (~fact_sales['date_key'].isin(dim_date['date_key'])).sum(),
        (~fact_sales['customer_key'].isin(dim_customer['customer_key'])).sum(),
        (~fact_sales['product_key'].isin(dim_product['product_key'])).sum()
    ]
})

referential_integrity['Status'] = (
    referential_integrity['Missing References'] == 0
)

referential_integrity

,Relationship,Missing References,Status
0,fact_sales → dim_date,0,True
1,fact_sales → dim_customer,0,True
2,fact_sales → dim_product,0,True


### Validation

The referential integrity checks confirm that every foreign key stored in `fact_sales` has a corresponding primary key in its respective dimension table.

No missing references were identified between the fact and dimension tables, confirming that the dimensional model is internally consistent.

## 4.9 Save the Analytical Tables

The dimensional model has now been built and validated. The final analytical tables are saved to the processed-data directory so they can be reused in the SQL analytics phase without repeating the transformation steps.

In [43]:

output_path = Path("../data/processed")
output_path.mkdir(parents=True, exist_ok=True)

In [44]:
dim_date.to_csv(
    output_path / "dim_date.csv",
    index=False
)

dim_customer.to_csv(
    output_path / "dim_customer.csv",
    index=False
)

dim_product.to_csv(
    output_path / "dim_product.csv",
    index=False
)

fact_sales.to_csv(
    output_path / "fact_sales.csv",
    index=False
)

In [45]:
list(output_path.glob("*.csv"))

[WindowsPath('../data/processed/dim_customer.csv'),
 WindowsPath('../data/processed/dim_date.csv'),
 WindowsPath('../data/processed/dim_product.csv'),
 WindowsPath('../data/processed/fact_sales.csv'),
 WindowsPath('../data/processed/scanner_data_clean.csv')]

## 4.10 Analytical Data Model Overview

The dimensional model is now complete. The cleaned transaction data has been organized into one central fact table and three supporting dimensions, with surrogate keys used to establish the relationships between them.

| Table | Purpose | Rows | Key Identifier |
|---|---|---:|---|
| `dim_date` | Calendar dimension | 363 | `date_key` |
| `dim_customer` | Customer dimension | 22,625 | `customer_key` |
| `dim_product` | Product dimension | 5,242 | `product_key` |
| `fact_sales` | Retail sales transaction line items | 131,706 | `transaction_id + product_key` |

 `fact_sales` has transaction-line-item grain. `transaction_id` alone is not unique because a transaction can contain multiple products.

### Phase Result

The analytical model has been successfully built and validated.

The fact table preserves the original transaction-line-item grain, while the dimension tables provide reusable context for time, customers, and products. Row-count, completeness, grain, and referential-integrity checks confirm that the model is internally consistent.

The completed analytical tables are now ready for the SQL analytics phase.